# Phase 01 — Bridge: your code → the HuggingFace stack

**Runs entirely on the Mac. No GPU needed.**

Companion to [`docs/phases/phase-01.md`](../docs/phases/phase-01.md). Deliverable: [`docs/BRIDGE.md`](../docs/BRIDGE.md).

---

## Why this notebook exists

You have already hand-written a GPT-2 124M training loop, LoRA, and multi-head attention. Almost nobody learning to fine-tune has done that.

So the fastest route is **not** a tutorial. It is to **diff your own code against the library** — because then every library feature arrives as an answer to a question you already know how to ask.

Carry one idea through the whole notebook:

> **The library's extra parameter is usually somebody else's bug report.**

Every option in `LoraConfig` exists because a default broke for someone. When you meet one, ask *what failure made this configurable?*

## How to use this

Cells marked **▶ RUN** are plumbing — just run them.
Cells marked **✍️ YOU** are yours. **Write your answer before running the cell below it.** Predicting first is the whole point (PLAN.md rule 2).

## Setup

**▶ RUN.** Nothing downloads yet — `AutoConfig` fetches only a small `config.json` per model, not weights.

In [1]:
import os
import sys

import torch
import torch.nn as nn
from transformers import AutoConfig

# Repo root derived from this notebook's location, so it works wherever Jupyter started.
REPO = os.path.abspath("..")
SRC = os.path.join(REPO, "src")

# There is no pyproject.toml, so the package is NOT pip-installed. The `!` cells further
# down run `python -m llm_finetuning_from_zero_to_hero...` and need src/ on PYTHONPATH.
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")

# Use THIS kernel's interpreter in every `!` cell, never a bare `python`.
# Exactly the trap from LESSONS.md L1: a bare `python` resolves through PATH and can be a
# different interpreter from the one running this notebook -- with different packages.
PY = sys.executable

import peft
import transformers

print("python      ", sys.version.split()[0])
print("executable  ", PY)
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("peft        ", peft.__version__)
print("device      ", "mps" if torch.backends.mps.is_available() else "cpu")
print("PYTHONPATH  ", SRC)

/Users/in04844/.pyenv/versions/FT_VENV/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python       3.12.10
executable   /Users/in04844/.pyenv/versions/FT_VENV/bin/python
torch        2.14.0
transformers 5.16.1
peft         0.20.0
device       mps
PYTHONPATH   /Users/in04844/Anant_Learning/LLM_fine_tuning_from_zero_to_hero/src


---
# Part 1 — Architecture: your GPT-2 vs Llama-3

### ✍️ YOU — predict first

Before running anything, write what you expect. You built GPT-2, so you know its shape.

| Question | Your answer |
| --- | --- |
| How many attention heads does Llama-3.1-8B have? | |
| How many **key/value** heads? | |
| Does it use learned position embeddings, like GPT-2? | |
| How many matrices in its MLP block? (GPT-2 has 2) | |
| Roughly how many parameters are in the embedding layer alone? | |

Then run the next cell.

In [6]:
# ▶ RUN — fetch and compare the architecture configs.
MODELS = {
    "Llama-3.1-8B":  "meta-llama/Llama-3.1-8B-Instruct",
    "Mistral-7B":    "mistralai/Mistral-7B-Instruct-v0.3",
    "Qwen3-1.7B":    "Qwen/Qwen3-1.7B",
    "Qwen3-0.6B":    "Qwen/Qwen3-0.6B",
    "GPT-2 (yours)": "gpt2",
}

configs = {}
for label, repo in MODELS.items():
    try:
        configs[label] = AutoConfig.from_pretrained(repo)
    except Exception as exc:                      # gated repos need `hf auth login`
        print(f"  SKIP {label}: {type(exc).__name__}")


def describe(cfg):
    """Pull the fields we care about, tolerating naming differences across families."""
    g = lambda *names: next((getattr(cfg, n) for n in names if hasattr(cfg, n)), None)
    return {
        "layers":     g("num_hidden_layers", "n_layer"),
        "d_model":    g("hidden_size", "n_embd"),
        "d_ff":       g("intermediate_size"),
        "q_heads":    g("num_attention_heads", "n_head"),
        "kv_heads":   g("num_key_value_heads"),
        "vocab":      g("vocab_size"),
        "tied_embed": g("tie_word_embeddings"),
    }


rows = {k: describe(v) for k, v in configs.items()}
hdr = f"{'model':<16}" + "".join(f"{c:>11}" for c in next(iter(rows.values())))
print(hdr)
print("-" * len(hdr))
for label, r in rows.items():
    print(f"{label:<16}" + "".join(f"{str(v):>11}" for v in r.values()))

model                layers    d_model       d_ff    q_heads   kv_heads      vocab tied_embed
---------------------------------------------------------------------------------------------
Llama-3.1-8B             32       4096      14336         32          8     128256      False
Mistral-7B               32       4096      14336         32          8      32768      False
Qwen3-1.7B               28       2048       6144         16          8     151936       True
Qwen3-0.6B               28       1024       3072         16          8     151936       True
GPT-2 (yours)            12        768       None         12       None      50257       True


## 📖 Reference — what each config field means, and where it sits

These seven numbers **completely determine the shape of the model.** Every VRAM calculation, every parameter count, every KV-cache estimate in this curriculum is derived from them. Worth knowing cold.

### Where each one appears in the forward pass

```text
input: token ids                                   [batch, seq]
  │
  ├─ embed_tokens          weight [vocab, d_model]        ← vocab, d_model
  ▼
hidden states                                      [batch, seq, d_model]
  │                                                  ↑ "the residual stream"
  │
  ├════════════ repeated `layers` times ═════════════════╗   ← layers
  ║                                                      ║
  ║   ├─ RMSNorm                                         ║
  ║   ├─ Self-attention                                  ║
  ║   │    q_proj  [d_model, q_heads  × head_dim]        ║   ← q_heads
  ║   │    k_proj  [d_model, kv_heads × head_dim]        ║   ← kv_heads
  ║   │    v_proj  [d_model, kv_heads × head_dim]        ║   ← kv_heads
  ║   │    o_proj  [q_heads × head_dim, d_model]         ║
  ║   ├─ + residual                                      ║
  ║   ├─ RMSNorm                                         ║
  ║   ├─ MLP (SwiGLU)                                    ║
  ║   │    gate_proj [d_model, d_ff]                     ║   ← d_ff
  ║   │    up_proj   [d_model, d_ff]                     ║   ← d_ff
  ║   │    down_proj [d_ff, d_model]                     ║
  ║   └─ + residual                                      ║
  ╚══════════════════════════════════════════════════════╝
  │
  ├─ final RMSNorm
  ├─ lm_head               weight [d_model, vocab]        ← tied_embed decides whether
  ▼                                                          this is a NEW matrix or
logits                                             [batch, seq, vocab]    embed_tokens reused
```

### The seven fields

| Field | HF name | What it is |
| --- | --- | --- |
| **`layers`** | `num_hidden_layers` | How many identical transformer blocks are stacked. Depth. Llama-3.1-8B: **32** |
| **`d_model`** | `hidden_size` | The width of the **residual stream** — every token is a vector of this size travelling through the whole network. The single most important number in the model. Llama-3.1-8B: **4096** |
| **`d_ff`** | `intermediate_size` | The width *inside* the MLP. Each block expands `d_model → d_ff`, applies a non-linearity, and contracts back. Llama-3.1-8B: **14336** (3.5×) |
| **`q_heads`** | `num_attention_heads` | How many attention heads ask questions ("what should I attend to?"). Llama-3.1-8B: **32** |
| **`kv_heads`** | `num_key_value_heads` | How many heads provide the answers. **Fewer than `q_heads` = Grouped-Query Attention** — several Q heads share one K/V pair. Llama-3.1-8B: **8** |
| **`vocab`** | `vocab_size` | Number of distinct tokens the tokenizer can produce. Llama-3.1-8B: **128256** |
| **`tied_embed`** | `tie_word_embeddings` | Whether `lm_head` **reuses** the embedding matrix instead of having its own. GPT-2: `True`. Llama-3.1-8B: `False` |

### One derived number you need constantly

```text
head_dim = d_model / q_heads          Llama-3.1-8B:  4096 / 32 = 128
```

Note it is divided by **`q_heads`**, not `kv_heads`. The K and V projections are *narrower* than Q — that is exactly what GQA does, and where its savings come from.

### Why each one matters to you

- **`layers` × (attention + MLP)** = the trunk parameter count → how much VRAM the weights need
- **`d_model`** appears in every matrix, so it drives parameter count *quadratically*
- **`d_ff`** is why ~80% of a layer's parameters live in the MLP — the reason LoRA targets all linear layers, not just attention (Phase 06)
- **`kv_heads`** sets your KV cache size and therefore how many concurrent requests you can serve (Phase 16)
- **`vocab`** × `d_model` × 4 bytes is the fp32 logits tensor — often the single largest allocation in training (`NUMBERS.md` §3)
- **`tied_embed=False`** costs an extra `vocab × d_model` matrix — 525 M parameters on Llama-3.1-8B

### What GQA actually is

Plain multi-head attention (your GPT-2) gives every query head its own key and value head:

```text
MHA   (GPT-2):    Q1 Q2 Q3 Q4  ...  32 queries
                  K1 K2 K3 K4  ...  32 keys     ← one each
                  V1 V2 V3 V4  ...  32 values

GQA   (Llama):    Q1 Q2 Q3 Q4   Q5 Q6 Q7 Q8  ...  32 queries
                  \___ K1 ___/  \___ K2 ___/  ...   8 keys    ← shared, 4 queries each
                  \___ V1 ___/  \___ V2 ___/  ...   8 values
```

The KV cache stores K and V **per `kv_head`**, not per `q_head`. Going 32 → 8 makes the cache **4× smaller**, which is the single biggest inference-cost lever in modern LLM design. Quality loss is small; the memory saving is enormous.

### ✍️ YOU — read the table

**1. `q_heads` vs `kv_heads`.** GPT-2 shows `None` for `kv_heads` because it predates the idea; it has one K and one V per Q head. Llama has 32 Q heads and 8 KV heads — that is **Grouped-Query Attention**.

*Your turn:* the KV cache stores one K and one V per **kv_head** per token. If Llama had 32 kv_heads instead of 8, how many times larger would the cache be? What does that do to the number of concurrent requests you can serve on 21.75 GiB?

> **Answer — 4× larger, and it costs you 4× the concurrency.**
>
> The formula: `KV bytes/token = 2 × layers × kv_heads × head_dim × bytes_per_value`
> (the leading 2 is because you cache both K *and* V).
>
> | | kv_heads | KV per token |
> | --- | --- | --- |
> | Llama-3.1-8B as built | 8 | `2 × 32 × 8 × 128 × 2` = **128 KiB** |
> | If it used plain MHA | 32 | `2 × 32 × 32 × 128 × 2` = **512 KiB** |
>
> After an 8B in bf16 (~16 GiB) you have roughly **5.75 GiB** left on a 21.75 GiB card.
>
> - at 128 KiB/token → ~47,000 cached tokens → **~11 concurrent requests at 4k context**
> - at 512 KiB/token → ~11,700 cached tokens → **~2 concurrent requests**
>
> So GQA is the difference between serving eleven users and serving two, on identical hardware. This is why it is in every modern model — and it is the arithmetic you will actually do in Phase 16.

**2. `d_ff / d_model`.** Compute that ratio for GPT-2 and for Llama. GPT-2's MLP is 2 matrices (`up`, `down`); Llama's is 3 (`gate`, `up`, `down`) — that is **SwiGLU**.

*Your turn:* if the MLP has 3 matrices of size `d_model × d_ff` and attention has 4 of size roughly `d_model × d_model`, where do most of the parameters live? Why does that matter when choosing LoRA `target_modules`?

> **Answer — ~81% of every layer lives in the MLP.**
>
> Llama-3.1-8B, per layer (`d=4096`, `d_ff=14336`, `head_dim=128`, `kv_heads=8`):
>
> | Matrix | Shape | Params |
> | --- | --- | ---: |
> | `q_proj` | 4096 × 4096 | 16.8 M |
> | `k_proj` | 4096 × **1024** | 4.2 M |
> | `v_proj` | 4096 × **1024** | 4.2 M |
> | `o_proj` | 4096 × 4096 | 16.8 M |
> | **attention total** | | **41.9 M** |
> | `gate_proj` | 4096 × 14336 | 58.7 M |
> | `up_proj` | 4096 × 14336 | 58.7 M |
> | `down_proj` | 14336 × 4096 | 58.7 M |
> | **MLP total** | | **176.2 M** |
> | **per layer** | | **218.1 M** → MLP is **81%** |
>
> Note `k_proj` and `v_proj` are only 1024 wide (`kv_heads × head_dim` = 8 × 128), not 4096. GQA shrinks the *weights* as well as the cache.
>
> **Why it decides `target_modules`:** attention-only LoRA (`q_proj`, `v_proj` — what most tutorials show) can only adapt ~19% of each layer's capacity. Targeting all linear layers reaches all of it. This is the arithmetic behind the Phase 06 finding that all-linear usually beats attention-only — and the intuition that attention modules shift *style* while the MLP holds *knowledge*.
>
> `d_ff/d_model`: GPT-2 = 4.0 (2 matrices), Llama = 3.5 (3 matrices). SwiGLU uses a smaller expansion precisely because it has a third matrix.

**3. `tied_embed`.** GPT-2 ties input and output embeddings; the 8B does not.

*Your turn:* what does untying cost in parameters? (Next cell computes it.)

> **Answer — 525 M parameters, about 1.0 GiB in bf16.**
>
> `vocab × d_model = 128256 × 4096 = 525.3 M`. Tied, `lm_head` reuses `embed_tokens` and costs nothing extra; untied, it is a second matrix that size.
>
> Combined, `embed_tokens` + `lm_head` = **1.05 B of the 8.03 B total**, i.e. **13% of the model is just vocabulary handling.** That is why Llama's 128k vocab is expensive and Mistral's 32k is not — compare their `total` rows in the next cell.
>
> It also matters directly in Phase 02: if you add a special token and only unfreeze `embed_tokens`, the model can *read* your token but can never *emit* it, because `lm_head` is a separate frozen matrix. That is what `modules_to_save=["embed_tokens", "lm_head"]` is for.

In [4]:
# ▶ RUN — parameter accounting, from the config alone. No weights downloaded.
def param_breakdown(cfg):
    d  = cfg.hidden_size
    ff = cfg.intermediate_size
    L  = cfg.num_hidden_layers
    V  = cfg.vocab_size
    kv = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
    head_dim = d // cfg.num_attention_heads

    # attention: q is d->d; k and v are d->(kv_heads*head_dim); o is d->d
    attn = d * d + 2 * (d * kv * head_dim) + d * d
    mlp  = 3 * d * ff                      # SwiGLU: gate, up, down
    per_layer = attn + mlp
    embed = V * d
    lm_head = 0 if getattr(cfg, "tie_word_embeddings", False) else V * d
    return {
        "attn/layer": attn, "mlp/layer": mlp, "per_layer": per_layer,
        "trunk": per_layer * L, "embed": embed, "lm_head": lm_head,
        "total": per_layer * L + embed + lm_head,
    }


for label in ("Llama-3.1-8B", "Mistral-7B", "Qwen3-1.7B"):
    if label not in configs:
        continue
    b = param_breakdown(configs[label])
    mlp_share = 100 * b["mlp/layer"] / b["per_layer"]
    print(f"\n{label}")
    for k, v in b.items():
        print(f"  {k:<12} {v/1e6:10.1f} M")
    print(f"  -> MLP is {mlp_share:.0f}% of each layer's parameters")


Llama-3.1-8B
  attn/layer         41.9 M
  mlp/layer         176.2 M
  per_layer         218.1 M
  trunk            6979.3 M
  embed             525.3 M
  lm_head           525.3 M
  total            8030.0 M
  -> MLP is 81% of each layer's parameters

Mistral-7B
  attn/layer         41.9 M
  mlp/layer         176.2 M
  per_layer         218.1 M
  trunk            6979.3 M
  embed             134.2 M
  lm_head           134.2 M
  total            7247.8 M
  -> MLP is 81% of each layer's parameters

Qwen3-1.7B
  attn/layer         12.6 M
  mlp/layer          37.7 M
  per_layer          50.3 M
  trunk            1409.3 M
  embed             311.2 M
  lm_head             0.0 M
  total            1720.5 M
  -> MLP is 75% of each layer's parameters


### ✍️ YOU

Does the `total` match the advertised size? Write the gap and explain it.

> **Answer — 8.03 B against an advertised "8B". The name is rounded down.**
>
> | Component | Params |
> | --- | ---: |
> | trunk (32 layers × 218.1 M) | 6.98 B |
> | `embed_tokens` | 0.53 B |
> | `lm_head` (untied) | 0.53 B |
> | **total** | **8.03 B** |
>
> Two things worth carrying forward:
>
> 1. **Marketing names are rounded.** "8B" is 8.03 B, "7B" is 7.24 B. When you compute VRAM at 16 bytes/param, a 0.03 B discrepancy is 0.5 GB — which matters when you are deciding whether something fits in 21.75 GiB.
> 2. **Compare Mistral-7B to Llama-3.1-8B.** Their *trunks* are nearly identical (both 32 layers, `d_model` 4096, `d_ff` 14336). Almost the entire size difference is vocabulary: Llama's 128k vs Mistral's 32k. Same architecture, different tokenizer, ~0.8 B parameters apart.
>
> That second point is why `NUMBERS.md` §3 calls the logits tensor a trap — `batch × seq × vocab × 4 bytes` is **4× larger** for Llama than Mistral at the same batch and sequence length, purely from vocab size.

The `MLP is N%` line is the arithmetic behind a LoRA default you will meet in Phase 06: **target all linear layers, not attention only.** Write why in one sentence.

> **Answer:** because ~81% of each layer's parameters sit in the MLP, so attention-only LoRA (`q_proj`, `v_proj` — the tutorial default) can adapt less than a fifth of the model's capacity, while all-linear reaches the part that actually holds the knowledge.

---
# Part 2 — LoRA: your `LinearWithLoRA` vs `peft`

The next cell is **your** implementation, transcribed from `learning_torch/04_learning_lora_from_scratch.ipynb`.

### ✍️ YOU — verify this is actually yours

Open that notebook side by side. If anything differs — the scaling, the shapes, the init — **edit this cell to match your version.** The comparison is worthless if I have quietly replaced your code with a copy of `peft`'s.

In [ ]:
class LinearWithLoRA(nn.Module):
    """A frozen linear layer plus a trainable low-rank update.

    out = W @ x  +  (alpha / r) * (B @ A @ x)
    """

    def __init__(self, base: nn.Linear, r: int, alpha: int):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False

        self.r, self.alpha = r, alpha
        self.scaling = alpha / r

        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)   # B stays zero (this is important)

    def forward(self, x):
        return self.base(x) + self.scaling * ((x @ self.lora_A.T) @ self.lora_B.T)


torch.manual_seed(0)
base = nn.Linear(64, 32, bias=False)
mine = LinearWithLoRA(base, r=8, alpha=16)

print(f"scaling            {mine.scaling}")
print(f"lora_A             {tuple(mine.lora_A.shape)}")
print(f"lora_B             {tuple(mine.lora_B.shape)}  (all zeros: {bool((mine.lora_B == 0).all())})")
print(f"trainable params   {sum(p.numel() for p in mine.parameters() if p.requires_grad)}")
print(f"formula r*(d_in+d_out) = 8*(64+32) = {8 * (64 + 32)}")

scaling            2.0
lora_A             (8, 64)
lora_B             (32, 8)  (all zeros: True)
trainable params   768
formula r*(d_in+d_out) = 8*(64+32) = 768


### ✍️ YOU — the zero-init question

`lora_B` starts at zero, so `B @ A = 0`, so the adapter is **exactly** a no-op at step 0.

*Your turn, before running:* what does that guarantee about the loss at step 0 of any LoRA fine-tune? What cheap sanity check does it give you for every real training run?

> **Answer — step-0 loss must equal the base model's loss on the same data. Exactly, not approximately.**
>
> At initialization `B = 0`, so `scaling × (B @ A @ x) = 0`, so `forward(x)` returns `base(x)` bit-for-bit. The adapter is not *approximately* neutral, it is *literally* neutral — which is what the cell below prints as `True`.
>
> Therefore the model at step 0 **is** the pretrained model, and its loss is the pretrained model's loss.
>
> **The sanity check:** evaluate the base model on a batch and note the loss. Start LoRA training and look at the first logged step. **They must match.** If they don't, one of these is true — and all four are common:
>
> - `target_modules` matched nothing — `print_trainable_parameters()` says `0.00%` (Phase 12 failure #5)
> - the adapter was built but never actually attached to the model
> - your eval and training paths use different tokenization or chat templates (Phase 02)
> - `init_lora_weights=False`, which disables zero-init and starts you somewhere random
>
> It costs one number and catches a whole class of silent failures **before** you spend three hours training. This is `PLAN.md` rule 5 — *inspect the tensors, not the loss curve* — in its cheapest possible form.
>
> **A second consequence:** LoRA can never start off *worse* than the base model. You begin at exactly the pretrained behaviour and move away from it deliberately. Compare a randomly-initialized head, which starts by actively damaging the output and has to climb back to par before it gains anything.
>
> ---
>
> #### Why `B` and not `A`? (the part almost nobody asks)
>
> Exactly **one** of the two must be zero, and it matters which.
>
> The adapter computes `B @ A @ x`, so the gradients are:
>
> ```text
> ∂L/∂A  ∝  Bᵀ · (upstream gradient)        ← scales with B
> ∂L/∂B  ∝  (A @ x)ᵀ · (upstream gradient)  ← scales with A
> ```
>
> - **If both `A` and `B` were zero:** both gradients are zero. And they stay zero, forever. The adapter is frozen at its initialization and **never learns anything** — while the loss curve looks like a perfectly healthy base model. A silent, total failure.
> - **With `A` random and `B` zero:** at step 0, `∂L/∂A = 0` (because `B = 0`) but `∂L/∂B ≠ 0` (because `A @ x ≠ 0`). So `B` moves first, becomes non-zero, and from the next step *both* matrices learn normally.
>
> That asymmetry is the whole trick: it buys an exact no-op at initialization **without** killing the gradient signal. Kaiming on `A`, zeros on `B`. Swapping them would work equally well; making them both zero would not.

In [9]:
# ▶ RUN — prove it.
x = torch.randn(4, 64)
print("adapter output identical to base output:", torch.equal(mine(x), base(x)))

adapter output identical to base output: True


In [11]:
# ▶ RUN — the same layer, built by peft, with YOUR A and B copied in.
from peft import LoraConfig, get_peft_model

base_copy = nn.Linear(64, 32, bias=False)
base_copy.weight.data.copy_(base.weight.data)
wrapper = nn.Sequential()
wrapper.add_module("target", base_copy)

peft_model = get_peft_model(
    wrapper,
    LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none", target_modules=["target"]),
)
peft_model.eval()
layer = peft_model.base_model.model.target

with torch.no_grad():                     # give B a non-zero value or this is trivial
    b_val = torch.randn_like(mine.lora_B) * 0.02
    mine.lora_B.copy_(b_val)
    layer.lora_A["default"].weight.copy_(mine.lora_A)
    layer.lora_B["default"].weight.copy_(b_val)

print("peft scaling      ", layer.scaling["default"], " (yours:", mine.scaling, ")")
print("max abs difference", (mine(x) - peft_model(x)).abs().max().item())

peft scaling       2.0  (yours: 2.0 )
max abs difference 0.0


### ✍️ YOU — what does `peft` add, and what failure does each answer?

Read `peft/tuners/lora/layer.py` alongside. Find the path with:

```python
import peft, os; print(os.path.dirname(peft.__file__))
```

This table becomes section 2 of `docs/BRIDGE.md`:

| `peft` feature | What failure made this configurable? | Do I need it? |
| --- | --- | --- |
| **`lora_B` zero-init** | *"How do I guarantee the adapter changes nothing at step 0?"* Without it you start by actively damaging a pretrained model and must climb back to par. | **Always on.** It is the default and you should never disable it. Gives you the step-0 loss check above |
| **`use_rslora`** (α/√r) | *"Why does raising the rank past ~64 stop helping — or start hurting?"* With plain `α/r`, doubling `r` halves the scaling, so the update shrinks exactly as fast as capacity grows. The two cancel. `α/√r` shrinks more slowly, so extra rank actually buys something. | **Yes, from Phase 06** — the moment you ablate `r ∈ {8,16,64,256}`. At r=16 the difference is small; at r=256 it decides whether the run is worth anything |
| **`init_lora_weights="pissa"` / `"loftq"`** | *"Can I start `A` somewhere better than random noise?"* **PiSSA** initializes from the principal singular vectors of `W`, so the adapter begins aligned with the directions that already matter — faster convergence. **LoftQ** initializes `A` and `B` to *compensate for the error introduced by 4-bit quantization*. | **LoftQ: yes for QLoRA** (Phase 04/06) — it directly attacks quantization error. PiSSA: optional, worth one ablation row |
| **`modules_to_save=["embed_tokens","lm_head"]`** | *"I added a special token, resized the embeddings, and the model can read it but will never emit it. Why?"* Because LoRA froze `lm_head`. The model has an input vector for your token but no output row to produce it. | **Yes, Phase 02** — the moment you add a custom token. Note it trains those modules **fully**, not with LoRA, so they are not small |
| **`merge_and_unload()`** | *"How do I ship this as one model instead of base + adapter?"* Folds `B @ A × scaling` into `W` and returns a plain `nn.Linear`. | **Yes, Phase 16.** With a caveat: merging a **QLoRA** adapter into a **bf16** base leaves residual error, because the adapter was trained against *quantized* weights it no longer sees |
| **`disable_adapter()`** | *"How do I get the base model's output without loading a second copy of it?"* A context manager that switches the adapter off, then back on. | **Yes, Phase 13 — and it is load-bearing.** See below |
| **`target_modules`** | *"Which layers should I even adapt?"* Attention-only was the original paper's choice; all-linear is the current default. | **Yes, Phase 06 ablates it.** From Part 1: the MLP holds 81% of the parameters, so attention-only reaches under a fifth of the model |

---

### The two worth dwelling on

**`disable_adapter()` — why it decides whether DPO fits on your hardware.**

DPO needs two models: the one being trained, and a frozen *reference* to measure divergence from. Naively that is two copies of Llama-3.1-8B in bf16 = **~32 GiB**, against your measured 21.75 GiB. It does not fit.

But a LoRA-wrapped model already *contains* both. Adapter on = the policy; adapter off = the reference. `DPOTrainer` calls `disable_adapter()` to get reference log-probabilities from the same weights:

```python
policy_logits = model(batch)                    # adapter active
with model.disable_adapter():
    ref_logits = model(batch)                   # adapter off = the original model
```

One set of weights, two behaviours, **~16 GiB instead of 32**. That single context manager is the difference between Phase 13 running on your box and not running at all. The cell below shows it working.

**`modules_to_save` — Phase 02's hardest lesson, in one line.**

Adding a token touches two matrices, and LoRA freezes both by default:

```text
embed_tokens [vocab, d_model]   ← the model READS your token here
lm_head      [d_model, vocab]   ← the model EMITS your token here
```

Resize and unfreeze only `embed_tokens` and you get a model that understands your special token perfectly and is structurally incapable of producing it. The loss looks fine. Generation never emits it. `modules_to_save` unfreezes both.

In [12]:
# ▶ RUN — see disable_adapter() and merging for yourself.
with torch.no_grad():
    active = peft_model(x)
    with peft_model.disable_adapter():
        disabled = peft_model(x)

print("adapter active  vs base:", (active - base(x)).abs().max().item())
print("adapter disabled vs base:", (disabled - base(x)).abs().max().item())
print("\n^ the second is ~0: one model, two behaviours, no second copy in memory.")

merged = peft_model.merge_and_unload()
print("\nafter merge, still matches yours:",
      torch.allclose(mine(x), merged(x), atol=1e-6))
print("merged module type:", type(merged.target).__name__, "<- plain nn.Linear again")

adapter active  vs base: 0.18698780238628387
adapter disabled vs base: 0.0

^ the second is ~0: one model, two behaviours, no second copy in memory.

after merge, still matches yours: True
merged module type: Linear <- plain nn.Linear again


---
# Part 3 — The training loop

> ### ⚠️ Correction (2026-09-14)
>
> Earlier versions of this notebook said *"diff your GPT-2 training loop against `Trainer`."* **That was wrong, and it was my error.**
>
> `ChatGPT2_124M_Pararameter_training/` contains the GPT-2 **architecture** — 276 lines in `src/xmer_arch/` (attention, MLP, layer, model), plus HF weight loading and verification to `<0.0001`. Excellent work, and exactly the right asset for **Part 1**. But it has **no optimizer, no `backward()`, and no step loop.** Its own README describes it as *"a faithful reproduction… matching HuggingFace's implementation"* — inference and verification, not training.
>
> **You have never written a training loop.** So Part 3 is not a bridge. It is new ground, and `train_min.py` is your **first** one.
>
> One real connection survives, though. Your `gpt2.py` has:
>
> ```python
> loss = nn.functional.cross_entropy(..., ignore_index=-1)
> ```
>
> That `ignore_index` **is** the loss-masking mechanism Phase 03 is built on — you have already touched the idea. Note the value: you used `-1`; HuggingFace's convention is **`-100`**. Same concept, different sentinel, and confusing the two means silently training on tokens you meant to exclude.

`src/.../train/train_min.py` is a complete, minimal training loop on a real tokenizer and model, deliberately **without** `Trainer`. Read it top to bottom — it is ~250 lines and every line is doing something you will meet again.

### ✍️ YOU — read the loop, then answer

Open [`train_min.py`](../src/llm_finetuning_from_zero_to_hero/train/train_min.py) and find these five things. They are the skeleton of *every* training loop you will ever write:

| # | Find the line that… | Why it is there |
| --- | --- | --- |
| 1 | builds `labels` with `-100` for prompt tokens | ← this is `ignore_index`, the HF convention. Your GPT-2 used `-1` |
| 2 | appends `tokenizer.eos_token_id` to the completion | without it the model never learns to stop |
| 3 | divides the loss by `grad_accum` | omit it and your loss silently rescales with the accumulation setting |
| 4 | calls `clip_grad_norm_` | the defence against one bad batch spiking the loss |
| 5 | steps the *scheduler* as well as the optimizer | warmup protects the first steps, when gradients are worst |

*Your turn:* of those five, **which would fail silently** if you left it out, and which would fail loudly?

> **Answer:**
>
> | # | If omitted | Fails… |
> | --- | --- | :---: |
> | 1 — `-100` masking | model trains on the prompt too; learns to imitate questions rather than answer them. Loss curve looks perfect | 🔇 **silent** |
> | 2 — EOS in labels | model never emits a stop token; generates until it hits the length cap, forever | 🔇 **silent** |
> | 3 — `/grad_accum` | loss is scaled by a constant; runs with different accumulation stop being comparable | 🔇 **silent** |
> | 4 — gradient clipping | a loss spike you can see on the curve | 🔊 loud |
> | 5 — scheduler step | LR stays at its initial value; you notice it in the logged `lr` column | 🔊 loud-ish |
>
> **Three of five fail silently**, and all three are about *data plumbing* rather than optimization. That is the shape of this whole field: the optimizer math is well-trodden and the libraries get it right; the failures that cost you days are in how the tokens were assembled and which ones carry loss.
>
> This is why Phases 02 and 03 exist, and why they come **before** Phase 04 trains anything real.

Now run the inspector below. It prints one training example as `token | id | label | in_loss?` and exits without training.

**Look for exactly two things:**
1. prompt tokens show `.` in `in_loss` — excluded (item 1 above)
2. the final `EOS` token shows `YES` — **included** (item 2 above)

In [ ]:
# ▶ RUN — downloads Qwen3-0.6B (~1.2 GB) the first time.
# {PY} is this kernel's own interpreter; PYTHONPATH was set in the setup cell.
!{PY} -m llm_finetuning_from_zero_to_hero.train.train_min --steps 20 --inspect

In [ ]:
# ▶ RUN — 20 real steps. Watch loss, grad_norm, and the LR warming up then decaying.
!{PY} -m llm_finetuning_from_zero_to_hero.train.train_min --steps 20

### ✍️ YOU — the closing list

`train_min.py` ends by printing what it does **not** do. Paste that list into section 6 of `docs/BRIDGE.md`.

That list is the argument for using TRL in Phase 04 — not that the library is conceptually different from your loop, but that it does ~15 small things correctly that are each easy to get subtly wrong.

*Your turn:* pick the **three** from that list you would be most likely to get wrong yourself, and say why.

> **Answer — rank by failure mode, not by difficulty.**
>
> The useful question is not *"which is hardest to implement?"* but *"which would I get wrong **without noticing**?"* Sort the list that way and it splits cleanly:
>
> | Omission | How it fails |
> | --- | --- |
> | gradient checkpointing | 💥 OOM. Immediate, obvious, fixed in one line |
> | multi-GPU | 💥 Does not run |
> | chunked cross-entropy | 💥 OOM on the logits tensor (`NUMBERS.md` §3) |
> | quantization | 💥 Model does not fit |
> | eval / metrics / tracking | 💥 You can see there is nothing to look at |
> | sequence packing | 🐌 Just slower — visible in tok/s |
> | **chat templates** | 🔇 **Silent** |
> | **multi-turn loss masking** | 🔇 **Silent** |
> | **checkpoint / resume** | 🔇 **Silent** |
>
> The loud ones cost minutes. **The silent ones cost days**, because you spend that time debugging your data or hyperparameters while the bug is in the plumbing. So my three are all from the bottom group:
>
> ---
>
> **1. Multi-turn loss masking.** The single most dangerous item on the list.
>
> A wrong mask produces a **smooth, entirely believable, descending loss curve** — and a model that imitates users instead of answering them, or never emits a stop token. You cannot detect it from the loss, from the gradients, or from `nvidia-smi`. Only by printing the mask.
>
> Single-turn is easy. Multi-turn is where it breaks: mask *every* assistant turn, none of the user turns, and include each assistant turn's terminating token. Get the boundary off by one — say, by masking the assistant header along with the user text — and you have trained the model not to produce the very token that starts its own reply.
>
> This is why Phase 03 builds `mask_inspector.py` before Phase 04 trains anything, and why `PLAN.md` rule 5 is *inspect the tensors, not the loss curve*.
>
> **2. Chat templates — specifically, train/inference mismatch.**
>
> `train_min.py` sidesteps this by using raw `prompt + completion`. Real models expect a precise format: `<|begin_of_text|><|start_header_id|>user<|end_header_id|>...` for Llama, `[INST]...[/INST]` for Mistral, `<|im_start|>user\n...` for Qwen.
>
> The failure mode is nasty because **training looks perfect.** The loss drops, eval on your held-out set looks great — because your eval uses the same wrong template. Then you serve the model, vLLM applies the *tokenizer's* template, and quality collapses. This is Phase 12 failure #9 and the **#1 real-world cause** of "it worked in testing."
>
> Two specific traps I would personally hit: **double BOS** (the template adds `<|begin_of_text|>`, then you tokenize with `add_special_tokens=True` and get two), and `pad_token = eos_token` masking genuine EOS tokens out of the loss along with the padding.
>
> **3. Checkpoint / resume.**
>
> I would save `model.state_dict()`, feel organised, and be wrong. A correct resume needs **four** things: model weights, optimizer state (Adam's two moment tensors — which are 8 bytes/param, larger than the model), the scheduler position, and the RNG + dataloader position.
>
> Miss the optimizer state and Adam restarts with zeroed momentum, producing a visible bump. Miss the RNG and data order, and the resumed run silently diverges from what an uninterrupted run would have produced. Neither raises an error, and `PLAN.md` rule 10 says a run you cannot re-create taught you nothing.
>
> ---
>
> **The meta-point:** three of my three are silent. The reason to use TRL is not that its authors are cleverer — it is that thousands of people have already hit these and the fixes are baked in. That is `LESSONS.md` **L0**: *the library's extra parameter is usually somebody else's bug report.*

---
# Done — now write it up

Phase 01 is complete when:

- [ ] `pytest tests/test_lora_equivalence.py -v` → **4 passed, 0 skipped**
- [ ] The `LinearWithLoRA` above genuinely matches your notebook (you checked)
- [ ] [`docs/BRIDGE.md`](../docs/BRIDGE.md) sections 1–6 filled in **in your own words**
- [ ] `Q01.1`–`Q01.5` in [`question_answer.md`](../question_answer.md) answered from memory
- [ ] You can name, cold, three things `Trainer` does that your loop did not

### Adversarial close (rule 12)

**What does this now fail on?** The equivalence test checks the *forward* pass only. It says nothing about:

- gradients — do `A` and `B` receive the same gradients as `peft`'s?
- `use_rslora` — the α/√r scaling
- merged vs unmerged numerical drift

Add whichever you can. Note the ones you cannot yet — Phase 06 returns to them.